# Parameter axes sweep

## Load the simulations

In [1]:
from ipynb_boilerplate import *
AGG = True
if AGG: configure_matplotlib(backend='Agg')
SAVE = True

SR = 0.2
PARAM_KEYS = ('Ra', 'Da')
PARAM_KEYS_TEX = PARAM_KEYS
PARAMS_FIXED = SYSTEM_A_REFERENCE.replace(sr=SR).remove(*PARAM_KEYS)

simulations_batch = GridSimulationFromNPZ.dict_from_dir_paths(
    PARAM_KEYS, 
    SIM_DIR_PATHS,
    ('c', 's'),
    ('f', 'mD', 'mC', 'uRMS'),
    PARAMS_FIXED,
    lazy=True,
    sorting=lambda d: dict(sorted(d.items(), reverse=False)),
)
save_fig = partial(
    save_figure, 
    dir_path=DIR_FIGS, 
    prefix=f"{'_'.join(PARAM_KEYS)}_sr={SR}", 
    pickle=True,
    close=AGG,
    file_ext=('svg', 'pdf', 'png'),
) if SAVE else pass_anything

In [2]:
print('Before parameter batching')
for i in SIM_DIR_PATHS: print(i)
print('After parameter batching')
for i in simulations_batch.values(): print(i.dir_path)

Before parameter batching
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=1.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__8cdc0d7deee27009/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__f1e8b107b29abbbb/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=100.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__5e822ff46ee05fa4/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=1000.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__3823d90f6e03a6fd/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=500.0|Da=1.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__795c9d7125905bce/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=500.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__c4deca4f8fe27111/
./data__c_stabilization=None|c_limits=True/

## Select the simulations

In [4]:
Ra_targets = (100.0, 500.0, 1000.0, 2000.0)
Da_targets = (1.0, 10.0, 100.0, 1000.0)
simulations = {
    (Ra, Da): v for (Ra, Da), v in simulations_batch.items()
    if include_prm(Ra, Ra_targets) and include_prm(Da, Da_targets)
}

In [5]:
print('After parameter selection')
for i in simulations.values(): print(i.dir_path)

After parameter selection
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=1.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__8cdc0d7deee27009/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__f1e8b107b29abbbb/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=100.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__5e822ff46ee05fa4/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=1000.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__3823d90f6e03a6fd/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=500.0|Da=1.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__795c9d7125905bce/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=500.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__c4deca4f8fe27111/
./data__c_stabilization=None|c_limits=True/

## Line plots

In [ ]:
SKIP_AVG = False

t_targets = (1.0, 3.0, 6.0, 9.0)

fPeak = 0.08
fRatioDash = 0.1
fRatioMax = 0.4
fRatio_expnt = 2
fRatio_slc = slice(10, None)

time_window = (0, 120)
n_cols = len(Ra_targets)
n_rows = int(len(simulations) / len(Ra_targets))
create_mfig = partial(create_multifigure, n_cols=n_cols, n_rows=n_rows)
mfig_mCmD, axs_mCmD, _ = create_mfig()
mfig_fMass, axs_fMass, _ = create_mfig()
mfig_uRMS, axs_uRMS, _ = create_mfig()
if not SKIP_AVG:
    mfig_cPlus, axs_cPlus, _ = create_mfig()
    mfig_cMinus, axs_cMinus, _ = create_mfig()
    mfig_sPlus, axs_sPlus, _ = create_mfig()
    mfig_fRatio, axs_fRatio, _ = create_mfig()

imap = lambda ix, iy: int((n_rows - 1 - iy) * n_cols + ix)
for (Ra, Da), sim in simulations.items():
    Ra_index = Ra_targets.index(Ra)
    Da_index = Da_targets.index(Da)
    idx = imap(Ra_index, Da_index)
    ttl = f'$Ra={as_int_if_close(Ra)}$\n$Da={as_int_if_close(Da)}$'
    kws = dict(title=ttl, x_label='$t$', x_lims=time_window)
    zeta0, aspect = sim['zeta0', 'aspect']
    Lx = aspect * 1
    mC, mD, f, uRMS = sim['mC', 'mD', 'f', 'uRMS']
    fZeta0, fZetaPlus, fZetaMinus = f.split()
    fMass = -(1/Lx) * derivative(mC.value_series, mC.time_series)
    plot_line(mfig_mCmD, axs_mCmD[idx],
        [(mC.time_series, mC.value_series), (mD.time_series, mD.value_series)],
        legend_labels=[f'$m_C$', '$m_D$'],
        **kws,
    )
    plot_line(
        mfig_fMass, axs_fMass[idx],
        [
            (fZeta0.time_series, [-np.sum(i) for i in fZeta0.value_series]), 
            (mC.time_series, fMass),
        ],
        y_lims=(0, fPeak),
        legend_labels=['$-F$', '$F_m$'],
        **kws,
    )
    plot_line(mfig_uRMS, axs_uRMS[idx],
        (uRMS.time_series, uRMS.value_series),
        y_label='$\mathrm{rms}(\mathbf{u})$',
        **kws,
    )
    axs_uRMS[idx].set_yscale('log')
    if SKIP_AVG:
        continue
    s, c = sim['s', 'c']
    zeta0_index = as_index(c.mesh.y_axis, zeta0)
    slcPlus = slice(zeta0_index, None)
    slcMinus = slice(0, zeta0_index)
    cPlus = NPyConstantSeries(
        average_grid(c.series, ('x', 'y'), (':', slcPlus)), c.time_series, 'cPlus',
    )
    cMinus = NPyConstantSeries(
        average_grid(c.series, ('x', 'y'), (':', slcMinus)), c.time_series, 'cMinus',
    )
    sPlus = NPyConstantSeries(
        average_grid(s.series, ('x', 'y'), (':', slcPlus)), s.time_series, 'sPlus',
    )
    plot_line(mfig_cPlus, axs_cPlus[idx],
        (cPlus.time_series, cPlus.value_series),
        y_label=f'$c^+$',
        **kws,
    )
    plot_line(mfig_cMinus, axs_cMinus[idx],
        (cMinus.time_series, cMinus.value_series),
        y_label=f'$c^-$',
        **kws,
    )
    plot_line(mfig_sPlus, axs_sPlus[idx],
        (sPlus.time_series, sPlus.value_series),
        y_label=f'$s^+$',
        **kws,
    )
    tFine = fZeta0.time_series
    cPlusFine = resample(cPlus.value_series, cPlus.time_series, tFine)
    cMinusFine = resample(cMinus.value_series, cMinus.time_series, tFine)
    fRatio = [
        -np.sum(flx) / (cp - cm)**fRatio_expnt 
        for flx, cp, cm in zip(fZeta0.value_series, cPlusFine, cMinusFine, strict=True)
    ]
    plot_line(mfig_fRatio, axs_fRatio[idx],
        (tFine[fRatio_slc], fRatio[fRatio_slc]),
        y_label=f'$-F/(c^+ - c^-)^{fRatio_expnt}$',
        y_lims=(0, fRatioMax),
        **kws,
    )

names = ['mCmD', 'fMass', 'uRMS']
if not SKIP_AVG:
    names.extend(('cPlus', 'cMinus', 'sPlus', 'fRatio'))
_locals = locals()
mfigs = {
    f'{n}(t)': _locals[f'mfig_{n}'] for n in names
}
for name, mfig in mfigs.items():
    save_fig(name)(mfig)

: 

# Scatter plots

In [7]:
uRMS_onset = 1e-3
Ra_exclude = (100.0, )
Ra_points, Da_points = [], []
tOnsets, fPeaks, tPeaks = [], [], []

for (Ra, Da), sim in simulations.items(): 
    if Ra in Ra_exclude:
        continue
    uRMS, f = sim['uRMS', 'f']
    fZeta0, fZetaPlus, fZetaMinus = f.split()
    tOnset = when_geq_first(uRMS.value_series, uRMS.time_series, uRMS_onset, None)
    if tOnset is None:
        continue
    Ra_points.append(Ra)
    Da_points.append(Da)
    tOnsets.append(tOnset)
    fZeta0Post_time_series = [t for t in fZeta0.time_series if t >= tOnset]
    fZeta0Post_value_series = [
        -np.sum(f) for t, f in zip(fZeta0.time_series, fZeta0.value_series) if t >= tOnset
    ]
    fPeak = np.max(fZeta0Post_value_series)
    fPeaks.append(fPeak)
    tPeak = when_geq_first(fZeta0Post_value_series, fZeta0Post_time_series, fPeak, np.inf)
    tPeaks.append(tPeak)


kws = dict(
    size=200.0,
    cmap='plasma',
    x_label='$Ra$',
    y_label='$Da$',
)

fig, ax = plot_scatter(
    (Ra_points, Da_points, tOnsets), 
    title=f'$t_{{\mathrm{{onset}}}}=\mathrm{{argmin}}_t\{{\mathrm{{rms}}(\mathbf{{u}}):\mathrm{{rms}}(\mathbf{{u}})\geq {uRMS_onset}\}}$',
    **kws,
)
ax.set_xscale('log')
ax.set_yscale('log')
save_fig('tOnset')(fig)

fig, ax = plot_scatter(
    (Ra_points, Da_points, fPeaks), 
    title='$\max_{t\geq t_{{\mathrm{{onset}}}}}(-F)$',
    **kws,
)
ax.set_xscale('log')
ax.set_yscale('log')
save_fig('fPeak')(fig)

fig, ax = plot_scatter(
    (Ra_points, Da_points, tPeaks), 
    title=f'$t_{{\mathrm{{peak}}}}=\mathrm{{argmax}}_t\{{-F:t\geq t_{{\mathrm{{onset}}}}\}}$',
    **kws,
)
ax.set_xscale('log')
ax.set_yscale('log') 
save_fig('tPeak')(fig)

